# Experiment 3 — Module 2: Advanced Autoregressive Decoding & Context Sweeps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepanshu-yadav/EVO_2_PETase/blob/main/experiments/experiment_3_deep_exploration/module_2_decoding_sweeps/exp3_module_2_decoding_and_scaling_sweeps.ipynb)

### Scientific Objectives:
1. **Novelty–Validity Pareto Frontier:** Sweep decoding temperatures $T \in \{0.6, 0.8, 1.0, 1.1\}$ and enable min-$p$ sampling (`min_p = 0.05`, `top_p = 0.92`) to maintain valid coding frames while maximizing sequence exploration.
2. **Context Depth Scaling & Memorization Cliff:** Test context scaling $n \in \{8, 16, 32, 64\}$ with $K = 50$ independent stratified draws to detect the exact context depth where sequence memorization spikes ($\overline{\max \text{PID}} > 80.0\%$ and novel yield drops by $> 25\%$).
3. **Contrastive Log-Likelihood Reranking:** Statistically isolate sequences leveraging thermophilic cutinase determinants over generic background esterases:
   $$\Delta \text{LL}(x) = \log P_{\text{Evo2}}(x \mid \text{Prompt}_{\text{thermo}}) - \log P_{\text{Evo2}}(x \mid \text{Prompt}_{\text{general}})$$ 

---
### Hardware Guidelines:
- **GPU:** NVIDIA A100 (40/80 GB) or L4 (24 GB). Standard T4 runtime is insufficient for Evo 2 7B.
- **Runtime:** Python 3 (High-RAM recommended).


## 1 · GPU Verification
Verify GPU availability. A100 or L4 is required.


In [ ]:
import subprocess
res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else "No GPU detected! Runtime -> Change runtime type -> A100 or L4 GPU.")


## 2 · Dual-Environment Setup

Colab's base kernel runs Python 3.13. `evo2-flash` requires `<3.13` and vLLM 0.26 requires a dedicated CUDA 12.9 PyTorch build.
To avoid kernel pollution or restart cycles, the engine runs inside an isolated Python 3.12 virtual environment (`/content/evo2env`), while data ingestion and downstream biophysical analysis run in the Colab kernel.


In [ ]:
#@title Install Evo 2 Flash Engine & MMseqs2 (~4-6 minutes)
import os, pathlib, subprocess

ENVDIR = "/content/evo2env"
VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"

!pip install -q uv
!uv venv -q --seed --python 3.12 {ENVDIR}

# PyTorch pinned to cu129 for vLLM compatibility
!{ENVDIR}/bin/pip install -q torch==2.11.0+cu129 torchvision==0.26.0+cu129 torchaudio==2.11.0+cu129 --extra-index-url https://download.pytorch.org/whl/cu129
!{ENVDIR}/bin/pip install -q "{VLLM_WHEEL}" --extra-index-url https://download.pytorch.org/whl/cu129
!{ENVDIR}/bin/pip install -q ninja evo2-flash

# Analysis libraries in the Colab kernel
!pip install -q biopython pandas numpy matplotlib seaborn openpyxl

# MMseqs2 static Linux AVX2 build
!wget -qO- https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz | tar xz


In [ ]:
#@title Verify Environment Integration
import os, shutil, subprocess

ENVDIR = "/content/evo2env"
PY = f"{ENVDIR}/bin/python"
os.environ.update({
    "VLLM_ENABLE_V1_MULTIPROCESSING": "0",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "PATH": f"{ENVDIR}/bin:/usr/local/cuda/bin:" + os.environ.get("PATH", ""),
})

MMSEQS = shutil.which("mmseqs") or os.path.abspath("mmseqs/bin/mmseqs")
assert os.path.exists(MMSEQS), "MMseqs2 binary not found! Re-run install cell."
print("MMseqs2 version:", subprocess.run([MMSEQS, "version"], capture_output=True, text=True).stdout.strip())

CHECK_CODE = """
import sys, torch, vllm, evo2_flash
print(f'Venv Python: {sys.version.split()[0]} | Torch: {torch.__version__} | vLLM: {vllm.__version__} | CUDA: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CUDA is not available inside the venv!'
"""
res = subprocess.run([PY, "-c", CHECK_CODE], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else res.stderr)
res.check_returncode()


## 3 · Data & Conditioned Prompt Ingestion
Load reference pool sequences (`petase_pool.fasta`, `petase_codons.json`) and the stratified prompt suites generated by Module 1.


In [ ]:
#@title Fetch or Load Prompt Suites and Reference Pool
import json, pathlib, urllib.request, shutil

DATA = pathlib.Path("data")
OUTPUTS = pathlib.Path("outputs")
DATA.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

REPO_BASE = "https://raw.githubusercontent.com/deepanshu-yadav/EVO_2_PETase/refs/heads/main/"

# 1. Download core pool if not running from clone
for name in ["petase_pool.fasta", "petase_codons.json"]:
    dest = DATA / name
    if not dest.exists():
        url = f"{REPO_BASE}/prepared_data/three_datasets_16nov2026/{name}"
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"Downloaded {name}")
        except Exception as e:
            print(f"Fetch failed for {name}: {e}")

# 2. Check for local Module 1 prompt outputs or download
PROMPT_FILES = [
    "prompts_thermo_curriculum.json",
    "prompts_thermo.json",
    "prompts_meso.json",
    "prompts_contrastive.json",
    "metadata_pool_annotated.tsv",
]

for pf in PROMPT_FILES:
    local_path = pathlib.Path(f"experiments/experiment_3_deep_exploration/outputs/{pf}")
    if local_path.exists():
        shutil.copy(local_path, OUTPUTS / pf)
        print(f"Loaded local {pf}")
    else:
        # Try downloading if hosted
        url = f"{REPO_BASE}/experiments/experiment_3_deep_exploration/outputs/{pf}"
        try:
            urllib.request.urlretrieve(url, OUTPUTS / pf)
            print(f"Downloaded {pf}")
        except:
            pass

# Verify pool records
from Bio import SeqIO
pool_records = list(SeqIO.parse(DATA / "petase_pool.fasta", "fasta"))
print(f"\nPool loaded: {len(pool_records)} proteins, length window: {min(len(r.seq) for r in pool_records)}-{max(len(r.seq) for r in pool_records)} aa")


## 4 · Experiment Configuration: Decoding Sweeps & Context Grid

Select hyperparameters:
- **Temperature Sweep:** $T \in \{0.6, 0.8, 1.0, 1.1\}$
- **Min-$p$ Filtering:** `min_p = 0.05`
- **Top-$p$ Truncation:** `top_p = 0.92`
- **Context Depth Grid:** $n \in \{8, 16, 32, 64\}$
- **Samples per Prompt:** 2 to 4 draws


In [ ]:
#@title Interactive Decoding Configuration

MODEL_NAME             = "evo2_7b"             #@param ["evo2_7b", "evo2_1b", "evo2_40b"]
PROMPT_SUITE_NAME      = "prompts_thermo_curriculum.json" #@param ["prompts_thermo_curriculum.json", "prompts_thermo.json", "prompts_meso.json"]

TEMPERATURES           = [0.6, 0.8, 1.0, 1.1]   # Temperature grid
MIN_P                  = 0.05                  # Min-p truncation
TOP_P                  = 0.92                  # Top-p nucleus cutoff
TOP_K                  = 4                     # Top-k bounding

SAMPLES_PER_PROMPT     = 2                     # Replicates per prompt draw (set 4 for production sweep)
MAX_CELLS              = 4                     # Context depths to execute (e.g. 4 for n=8, 16, 32, 64)
GPU_MEM_FRACTION       = 0.85                  # vLLM GPU memory fraction
SEED                   = 1337                  # Master RNG seed

SPACER                 = "GGGGGGGG"

suite_path = OUTPUTS / PROMPT_SUITE_NAME
assert suite_path.exists(), f"Prompt suite {suite_path} not found!"
suite_data = json.loads(suite_path.read_text())
active_cells = suite_data["cells"][:MAX_CELLS]

total_gens = sum(len(c["draws"]) * SAMPLES_PER_PROMPT for c in active_cells) * len(TEMPERATURES)
print(f"Configured Sweep: {len(active_cells)} cells x {len(TEMPERATURES)} temperatures x {SAMPLES_PER_PROMPT} samples = {total_gens} generations")


## 5 · Autoregressive Generation Stage
Dispatches the generation job to the dedicated Python 3.12 venv. Leverages continuous batching and automatic prefix caching via `evo2_flash`.


In [ ]:
#@title Execute Evo 2 Autoregressive Generation
import json, pathlib, subprocess, time

WORK = pathlib.Path("work")
WORK.mkdir(exist_ok=True)

# Prepare cells manifest for venv script
gen_manifest = {
    "model": MODEL_NAME,
    "gpu_memory_utilization": GPU_MEM_FRACTION,
    "temperatures": TEMPERATURES,
    "min_p": MIN_P,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "seed": SEED,
    "cells": [{
        "cell_name": c["cell_name"],
        "n_examples": c["n_examples"],
        "gen_length": c["gen_length"],
        "prompts": [d["prompt"] for d in c["draws"] for _ in range(SAMPLES_PER_PROMPT)],
        "draw_indices": [d["draw_idx"] for d in c["draws"] for _ in range(SAMPLES_PER_PROMPT)],
    } for c in active_cells]
}
(WORK / "sweep_manifest.json").write_text(json.dumps(gen_manifest, indent=2))

GEN_SCRIPT = """
import json, pathlib, time
from evo2_flash import Evo2

WORK = pathlib.Path('work')
manifest = json.loads((WORK / 'sweep_manifest.json').read_text())
cells = manifest['cells']

model = Evo2(manifest['model'], gpu_memory_utilization=manifest['gpu_memory_utilization'])
print(f'Evo 2 Engine online. max_model_len = {model.max_model_len:,}', flush=True)

all_results = {}
t0 = time.time()

for c_idx, c in enumerate(cells, 1):
    c_name = c['cell_name']
    n_ex = c['n_examples']
    gen_len = c['gen_length']
    prompts = c['prompts']
    all_results[c_name] = {}

    for temp in manifest['temperatures']:
        t_cell = time.time()
        out = model.generate(
            prompts,
            n_tokens=gen_len,
            temperature=temp,
            top_k=manifest['top_k'],
            top_p=manifest['top_p'],
            seed=manifest['seed'] + c_idx * 1000 + int(temp * 10),
            verbose=0,
        )
        tok_count = sum(len(p) for p in prompts)
        cached_pct = 100 * sum(out.num_cached_tokens) / max(tok_count, 1)
        elapsed = time.time() - t_cell
        print(f'[{c_idx}/{len(cells)}] {c_name} (T={temp}) | {len(prompts)} seqs in {elapsed:5.1f}s | Prefix cache: {cached_pct:4.1f}%', flush=True)
        all_results[c_name][str(temp)] = {
            'sequences': out.sequences,
            'draw_indices': c['draw_indices'],
            'n_examples': n_ex,
        }

(WORK / 'raw_sweep_continuations.json').write_text(json.dumps(all_results))
print(f'Generation complete in {time.time() - t0:.1f}s', flush=True)
"""
(WORK / "run_sweeps.py").write_text(GEN_SCRIPT)

print("Starting GPU Generation...")
!{PY} work/run_sweeps.py


## 6 · CDS Extraction, In-Frame Translation & Length Gating

Filters candidates across 3 physical gating criteria:
1. **Parsing:** Emits a valid closing `GGGGGGGG` separator.
2. **Frame & Translation:** Multiples of 3 nt, standard genetic code translation, 0 internal premature stop codons.
3. **Length Window:** Mature domain length falls in the canonical $241 \le L \le 306$ aa interval.


In [ ]:
#@title Parse Continuations & Translate to Mature Proteins
import json, pathlib, csv, warnings
from Bio.Seq import Seq
from Bio import BiopythonWarning
warnings.simplefilter("ignore", BiopythonWarning)

WORK = pathlib.Path("work")
raw_results = json.loads((WORK / "raw_sweep_continuations.json").read_text())

parsed_candidates = []
counter = 0

for c_name, temp_dict in raw_results.items():
    for temp_str, data in temp_dict.items():
        temp = float(temp_str)
        n_ex = data["n_examples"]
        seqs = data["sequences"]
        draws = data["draw_indices"]

        for s_idx, (cont, d_idx) in enumerate(zip(seqs, draws), 1):
            counter += 1
            design_id = f"EXP3_n{n_ex:03d}_T{temp:.1f}_s{counter:04d}"

            # Extract CDS
            clean_cont = cont.upper().strip()
            cds = clean_cont.split(SPACER)[0] if SPACER in clean_cont else None
            parsed_ok = (cds is not None)

            valid_cds = False
            mature_aa = ""
            reason = "unparsed_no_spacer"

            if parsed_ok and len(cds) >= 3:
                remainder = len(cds) % 3
                in_frame_cds = cds[:-remainder] if remainder else cds
                aa_raw = str(Seq(in_frame_cds).translate())
                mature_aa = aa_raw.rstrip("*")
                has_internal_stop = "*" in mature_aa
                in_window = (241 <= len(mature_aa) <= 306)

                if has_internal_stop:
                    reason = "internal_stop_codon"
                elif not in_window:
                    reason = f"length_{len(mature_aa)}_outside_window"
                else:
                    valid_cds = True
                    reason = "valid"

            parsed_candidates.append({
                "design_id": design_id,
                "cell": c_name,
                "n_examples": n_ex,
                "temperature": temp,
                "draw_idx": d_idx,
                "parsed_ok": parsed_ok,
                "valid_cds": valid_cds,
                "reason": reason,
                "length_aa": len(mature_aa),
                "sequence_aa": mature_aa,
                "cds_nt": cds or "",
            })

total = len(parsed_candidates)
parsed_count = sum(1 for c in parsed_candidates if c["parsed_ok"])
valid_count = sum(1 for c in parsed_candidates if c["valid_cds"])
print(f"Parsing Summary:")
print(f"  Total Generated:   {total}")
print(f"  Parsed Delimiter:  {parsed_count} ({100*parsed_count/total:.1f}%)")
print(f"  Valid Mature Fold: {valid_count} ({100*valid_count/total:.1f}%)")


## 7 · Contrastive Log-Likelihood Reranking ($\Delta \text{LL}$)

We evaluate log-likelihood of each valid sequence under thermophilic conditioning versus general background conditioning:
$$\Delta \text{LL}(x) = \log P_{\text{Evo2}}(x \mid \text{Prompt}_{\text{thermo}}) - \log P_{\text{Evo2}}(x \mid \text{Prompt}_{\text{general}})$$ 
Higher $\Delta \text{LL}$ isolates designs with elevated thermophilic regulatory signatures.


In [ ]:
#@title Compute Delta-LL Reranking with Evo 2
import subprocess, json

valid_designs = [c for c in parsed_candidates if c["valid_cds"]]
print(f"Scoring Delta-LL for {len(valid_designs)} valid designs...")

# Save designs and contrastive prompts for scoring
contrastive_path = OUTPUTS / "prompts_contrastive.json"
has_contrastive = contrastive_path.exists()

if has_contrastive and len(valid_designs) > 0:
    (WORK / "to_score.json").write_text(json.dumps({
        "designs": [{"id": c["design_id"], "cds": c["cds_nt"]} for c in valid_designs],
        "contrastive_file": str(contrastive_path),
    }))

    SCORE_SCRIPT = """
import json, pathlib
from evo2_flash import Evo2

WORK = pathlib.Path('work')
data = json.loads((WORK / 'to_score.json').read_text())
cont = json.loads(pathlib.Path(data['contrastive_file']).read_text())
pair = cont['pairs'][0]

model = Evo2('evo2_7b', gpu_memory_utilization=0.85)
thermo_p = pair['thermo_prompt']
general_p = pair['general_prompt']

thermo_inputs = [thermo_p + d['cds'] for d in data['designs']]
general_inputs = [general_p + d['cds'] for d in data['designs']]

scores_t = model.score_sequences(thermo_inputs, reduce_method='sum')
scores_g = model.score_sequences(general_inputs, reduce_method='sum')

delta_ll = {d['id']: round(float(st - sg), 3) for d, st, sg in zip(data['designs'], scores_t, scores_g)}
(WORK / 'delta_ll_scores.json').write_text(json.dumps(delta_ll))
print(f'Scored {len(delta_ll)} designs successfully.')
"""
    (WORK / "score_delta_ll.py").write_text(SCORE_SCRIPT)
    try:
        subprocess.run([PY, str(WORK / "score_delta_ll.py")], check=True)
        delta_dict = json.loads((WORK / "delta_ll_scores.json").read_text())
        for c in parsed_candidates:
            c["delta_ll"] = delta_dict.get(c["design_id"], 0.0)
        print("Delta-LL scores integrated into generation table.")
    except Exception as e:
        print(f"Scoring skipped or failed: {e}")
else:
    print("Contrastive suite not found or no valid designs to score.")


## 8 · MMseqs2 Homology Search & Novelty Quantification
Aligns all candidates against `data/petase_pool.fasta` to determine:
- **Maximum Percent Identity (max PID):** Identity against the closest natural homolog in the pool.
- **Novelty Gate:** Designs with $\max \text{PID} < 85.0\%$ are classified as novel.


In [ ]:
#@title MMseqs2 Homology & Alignment Search
import subprocess, csv, pathlib

WORK = pathlib.Path("work")
valid_designs = [c for c in parsed_candidates if c["valid_cds"]]

# Write FASTA of valid designs
query_faa = WORK / "exp3_valid_designs.faa"
with open(query_faa, "w") as f:
    for c in valid_designs:
        f.write(f">{c['design_id']}\n{c['sequence_aa']}\n")

target_faa = DATA / "petase_pool.fasta"
aln_m8 = WORK / "aln_vs_pool.m8"
tmp_dir = WORK / "tmp_mmseqs"
tmp_dir.mkdir(exist_ok=True)

MMSEQS_CMD = [
    MMSEQS, "easy-search",
    str(query_faa), str(target_faa), str(aln_m8), str(tmp_dir),
    "--format-output", "query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits",
    "-s", "7.5", "--max-seqs", "10", "-v", "0",
]
subprocess.run(MMSEQS_CMD, check=True)

# Parse best hits
best_hits = {}
with open(aln_m8) as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        qid, tid, fident = row[0], row[1], float(row[2]) * 100.0
        if qid not in best_hits or fident > best_hits[qid]["pid"]:
            best_hits[qid] = {"pid": fident, "target": tid}

for c in parsed_candidates:
    hit = best_hits.get(c["design_id"])
    if hit:
        c["max_pid"] = round(hit["pid"], 2)
        c["best_hit"] = hit["target"]
        c["is_novel"] = (hit["pid"] < 85.0)
    else:
        c["max_pid"] = 0.0
        c["best_hit"] = "none"
        c["is_novel"] = False

novel_count = sum(1 for c in valid_designs if c.get("is_novel", False))
print(f"Alignment Complete: {len(best_hits)} designs aligned to pool.")
print(f"Coherent Novel Yield (<85% PID): {novel_count}/{len(valid_designs)} ({100*novel_count/max(len(valid_designs),1):.1f}%)")


## 9 · Memorization Cliff Detection & Pareto Frontier Visualization

We evaluate:
1. **Memorization Cliff:** Does $\overline{\max \text{PID}} > 80.0\%$ with a $>25\%$ drop in novel yield at deeper contexts?
2. **Novelty–Validity Pareto Frontier:** Optimal trade-off between decoding temperature $T$ and structural validity.


In [ ]:
#@title Plot Memorization Cliff & Pareto Frontier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame(parsed_candidates)
valid_df = df[df["valid_cds"]].copy()

# 1. Memorization Cliff Analysis by Context Depth n
cliff_summary = []
for n, group in valid_df.groupby("n_examples"):
    mean_pid = group["max_pid"].mean()
    novel_pct = (group["max_pid"] < 85.0).mean() * 100.0
    cliff_summary.append({
        "n_examples": n,
        "mean_max_pid": mean_pid,
        "novel_yield_pct": novel_pct,
        "count": len(group),
    })
cliff_df = pd.DataFrame(cliff_summary).sort_values("n_examples")
print("Memorization Audit across Context Depths (n):")
display(cliff_df)

# 2. Multi-panel Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
sns.set_theme(style="whitegrid")

# Panel A: Context Scaling vs Novel Yield
ax1 = axes[0, 0]
sns.lineplot(data=valid_df, x="n_examples", y="max_pid", hue="temperature", marker="o", palette="viridis", ax=ax1)
ax1.axhline(80.0, color="red", linestyle="--", label="Cliff Threshold (80% PID)")
ax1.set_title("A: Context Depth (n) vs Mean Max Identity", fontsize=13, fontweight="bold")
ax1.set_xlabel("Context Depth (n)")
ax1.set_ylabel("Max % Identity vs Pool")
ax1.legend()

# Panel B: Temperature vs Novel Yield
ax2 = axes[0, 1]
t_summary = df.groupby("temperature").agg(
    parse_pct=("parsed_ok", lambda x: x.mean() * 100),
    valid_pct=("valid_cds", lambda x: x.mean() * 100),
    novel_pct=("is_novel", lambda x: x.mean() * 100),
).reset_index()
sns.barplot(data=t_summary, x="temperature", y="novel_pct", palette="mako", ax=ax2)
ax2.set_title("B: Decoding Temperature vs Novel Yield (<85% PID)", fontsize=13, fontweight="bold")
ax2.set_ylabel("Novel Yield (% of Generated)")

# Panel C: Identity Distribution
ax3 = axes[1, 0]
sns.histplot(data=valid_df, x="max_pid", hue="temperature", bins=25, kde=True, palette="turbo", ax=ax3)
ax3.axvline(85.0, color="darkred", linestyle="--", label="Novelty Threshold (85%)")
ax3.set_title("C: Homology Spectrum vs Natural Pool", fontsize=13, fontweight="bold")
ax3.set_xlabel("Max % Identity")
ax3.legend()

# Panel D: Delta-LL Thermophilic Enrichment
ax4 = axes[1, 1]
if "delta_ll" in valid_df.columns and valid_df["delta_ll"].nunique() > 1:
    sns.scatterplot(data=valid_df, x="max_pid", y="delta_ll", hue="temperature", size="length_aa", palette="coolwarm", ax=ax4)
    ax4.set_title("D: Contrastive Delta-LL vs Novelty", fontsize=13, fontweight="bold")
    ax4.set_ylabel("Delta-LL [log P(thermo) - log P(gen)]")
else:
    sns.lineplot(data=t_summary, x="temperature", y="valid_pct", marker="s", color="green", ax=ax4)
    ax4.set_title("D: Physical Validity vs Temperature", fontsize=13, fontweight="bold")
    ax4.set_ylabel("Valid In-Frame Folds (%)")

plt.tight_layout()
plt.savefig("novelty_validity_pareto_sweeps.png")
plt.show()
print("Saved figure to novelty_validity_pareto_sweeps.png")


## 10 · Export Deliverables & Candidate Screening Panel

Outputs:
- `generated_continuations_grid.tsv`: Comprehensive record of all generations and biophysical metrics.
- `designs_exp3_screened_novel.faa`: Mature amino acid FASTA of all coherent, novel designs clearing the 85% PID threshold, formatted directly for ESMFold structure prediction in Module 3.


In [ ]:
#@title Export TSV Tables & Screened Candidate FASTA
import pandas as pd, pathlib

OUTPUTS = pathlib.Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

grid_tsv = OUTPUTS / "generated_continuations_grid.tsv"
df = pd.DataFrame(parsed_candidates)
df.to_csv(grid_tsv, sep="\t", index=False)
print(f"Saved {len(df)} candidate records to {grid_tsv}")

# Screened novel candidates
novel_df = df[df["valid_cds"] & df["is_novel"]].copy()
if "delta_ll" in novel_df.columns:
    novel_df = novel_df.sort_values("delta_ll", ascending=False)

out_faa = OUTPUTS / "designs_exp3_screened_novel.faa"
with open(out_faa, "w") as f:
    for _, r in novel_df.iterrows():
        header = f"{r['design_id']} n={r['n_examples']} T={r['temperature']} max_pid={r['max_pid']}%"
        if "delta_ll" in r:
            header += f" delta_ll={r['delta_ll']}"
        f.write(f">{header}\n")
        seq = r["sequence_aa"]
        f.write("\n".join(seq[j:j+60] for j in range(0, len(seq), 60)) + "\n")

print(f"Saved {len(novel_df)} coherent, novel leads to {out_faa}")
print("Module 2 Pipeline execution complete!")
